## INSTALAÇÃO DAS BIBLIOTECAS AZURE-STORAGE-BLOB E PYSPARK

In [ ]:
pip install azure-storage-blob

In [2]:
pip install pyspark

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.4/281.4 MB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 KB 20.2 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.3.2-py2.py3-none-any.whl size=281824025 sha256=292ceb463bb386e4ecd51729b9b54a195da1434747cdb469dfd367150586817e
  Stored in directory: /root/.cache/pip/wheels/b1/59/a0/a1a0624b5e865fd389919c1a10f53aec9b12195d6747710baf
Successfully built pyspark


In [3]:
import requests
import azure.storage.blob

In [4]:
def download(url_arquivo, nome_arquivo):
  req = requests.get(url_arquivo)
  conteudo = req.content
  arquivo = open(nome_arquivo, 'wb')
  arquivo.write(conteudo)
  arquivo.close()

In [9]:
url_ordem_compra = 'https://datalakeaula687878.blob.core.windows.net/datalake/raw-zone/olist/olist_orders_dataset/olist_orders_dataset.csv'
url_cliente = 'https://datalakeaula687878.blob.core.windows.net/datalake/raw-zone/olist/olist_customers_dataset/olist_customers_dataset.csv'
nome_arquivo_ordem_compra = 'olist_orders_dataset.csv'
nome_arquivo_cliente = 'olist_customers_dataset.csv'
download(url_ordem_compra, nome_arquivo_ordem_compra)
download(url_cliente, nome_arquivo_cliente)

ConnectionError: ignored

In [6]:
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import *

In [7]:
spark = SparkSession.builder.getOrCreate()

In [8]:
schema_orders = StructType([ \
    StructField('order_id', StringType(), True), \
    StructField('customer_id', StringType(), True), \
    StructField('order_status', StringType(), True), \
    StructField('order_purchase_timestamp', TimestampType(), True), \
    StructField('order_approved_at', TimestampType(), True), \
    StructField('order_delivered_carrier_date', TimestampType(), True), \
    StructField('order_delivered_customer_date', TimestampType(), True), \
    StructField('order_estimated_delivery_date', TimestampType(), True) \
])

In [ ]:
df_orders = spark.read.csv(nome_arquivo_ordem_compra, header=True, inferSchema=True, schema=schema_orders)

In [ ]:
df_clientes = spark.read.csv(nome_arquivo_cliente, header=True, inferSchema=True)

In [ ]:
df_clientes.show(truncate=False, n=5)

In [ ]:
df_orders.show(truncate=False, n=3)

In [ ]:
df_orders.printSchema()

In [ ]:
# df_orders.agg({"order_purchase_timestamp": "max"}).show()

In [ ]:
df_orders.createOrReplaceTempView("df_orders")

In [ ]:
df_clientes.createOrReplaceTempView("df_clientes")

In [ ]:
df_clientes = spark.sql("""
SELECT  *,
        CASE
          WHEN customer_state IN ('MG', 'SP', 'RJ', 'ES') THEN 'Sudeste'
          WHEN customer_state IN ('RS', 'SC', 'PR') THEN 'Sul'
          WHEN customer_state IN ('DF', 'GO', 'MS', 'MT') THEN 'Centro-Oeste'
          WHEN customer_state IN ('AM', 'AP', 'RR', 'RO', 'PA', 'AC', 'TO') THEN 'Norte'
          ELSE 'Nordeste'
        END AS customer_region
FROM    df_clientes
""")

In [ ]:
df_clientes.createOrReplaceTempView("df_clientes_2")

In [ ]:
spark.sql("""
SELECT    customer_region,
          customer_state,
          COUNT(*) AS quantidade_clientes
FROM      df_clientes_2
GROUP BY  ROLLUP(customer_region,
          customer_state)
ORDER BY  customer_region,
          quantidade_clientes desc
""").show(truncate=False, n=1000)